
# Activation Functions in Neural Networks — Why Non-Linearity Matters

### PyTorch Demonstration Notebook

**Objective:**  
We will train two neural networks on the same **non-linearly separable `make_moons` dataset**:

1. **Without activation functions** — every layer is linear, so the entire network remains mathematically equivalent to a single linear transformation.
2. **With ReLU activation** — hidden layers become nonlinear and can learn the curved decision boundary.

### What we will visualize

- Original dataset
- Training loss
- Decision boundary of the **no-activation model**
- Decision boundary of the **ReLU model**
- Accuracy comparison
- Final theoretical conclusion

> **Important:** The dataset is intentionally non-linear so that the effect of activation functions is easy to see.



## 1. Import Libraries

We use **PyTorch for the neural networks and training**.  
`scikit-learn` is used only to generate the illustrative `make_moons` dataset and calculate the train/test split.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Create a Non-Linear Dataset

`make_moons` creates two curved classes. A single straight line cannot perfectly separate them.

In [ ]:

# Generate two interleaving half-moon classes
X, y = make_moons(
    n_samples=2000,
    noise=0.20,
    random_state=42
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test  = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


In [ ]:

plt.figure(figsize=(8, 6))

plt.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train.squeeze(),
    cmap="coolwarm",
    edgecolors="k",
    alpha=0.7
)

plt.title("Original Non-Linear Dataset — make_moons")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(alpha=0.25)
plt.show()



## 3. Why This Dataset Is Difficult for a Linear Model

The two classes form curved shapes:

- Class 0 → lower/outer moon
- Class 1 → upper/inner moon

A linear classifier can only learn a boundary of the form:

\[
w_1x_1+w_2x_2+b=0
\]

which is a **straight line**.

Therefore, a model containing only linear transformations should struggle with this dataset.



## 4. Neural Network WITHOUT Activation Function

Notice that there is **no ReLU, Sigmoid, Tanh, etc.**

```text
Input
  ↓
Linear(2 → 16)
  ↓
Linear(16 → 16)
  ↓
Linear(16 → 1)
  ↓
Output
```

Although this looks like a deep neural network, mathematically:

\[
W_3(W_2(W_1X+b_1)+b_2)+b_3
\]

can be simplified to:

\[
W'X+b'
\]

So the entire network is still just a **linear function**.


In [ ]:

class LinearOnlyNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.Linear(16, 16),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


linear_model = LinearOnlyNet().to(device)

print(linear_model)


## 5. Training Function

In [ ]:

def train_model(model, X_train, y_train, epochs=1000, lr=0.01):
    model = model.to(device)

    X_train_device = X_train.to(device)
    y_train_device = y_train.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []

    for epoch in range(epochs):
        model.train()

        logits = model(X_train_device)
        loss = criterion(logits, y_train_device)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return losses


In [ ]:

linear_losses = train_model(
    linear_model,
    X_train,
    y_train,
    epochs=1000,
    lr=0.01
)

print("Final training loss:", linear_losses[-1])


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(linear_losses)

plt.title("Training Loss — Network WITHOUT Activation")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.grid(alpha=0.25)
plt.show()



## 6. Evaluate the Linear-Only Model

Because there is no activation function, the network can only produce a **linear decision boundary**.


In [ ]:

def evaluate_model(model, X, y):
    model.eval()

    with torch.no_grad():
        logits = model(X.to(device))
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    accuracy = (predictions.cpu() == y).float().mean().item()
    return accuracy


linear_train_acc = evaluate_model(linear_model, X_train, y_train)
linear_test_acc = evaluate_model(linear_model, X_test, y_test)

print(f"Linear-only Train Accuracy: {linear_train_acc:.4f}")
print(f"Linear-only Test Accuracy : {linear_test_acc:.4f}")


In [ ]:

def plot_decision_boundary(model, X, y, title, ax=None):
    model.eval()

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    x_min, x_max = X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5
    y_min, y_max = X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )

    grid = torch.tensor(
        np.c_[xx.ravel(), yy.ravel()],
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        probabilities = torch.sigmoid(model(grid))
        predictions = (probabilities >= 0.5).float()

    Z = predictions.cpu().numpy().reshape(xx.shape)

    ax.contourf(
        xx, yy, Z,
        levels=[-0.5, 0.5, 1.5],
        alpha=0.20,
        cmap="coolwarm"
    )

    ax.contour(
        xx, yy, Z,
        levels=[0.5],
        linewidths=2
    )

    ax.scatter(
        X[:, 0],
        X[:, 1],
        c=y.squeeze(),
        cmap="coolwarm",
        edgecolors="k",
        alpha=0.75
    )

    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.grid(alpha=0.2)

    return ax


plot_decision_boundary(
    linear_model,
    X_test,
    y_test,
    "Decision Boundary — WITHOUT Activation"
)

plt.show()



### Observation

The decision boundary is essentially a **straight line**.

This demonstrates the key mathematical point:

> **Stacking linear layers without a nonlinear activation does NOT make the network nonlinear.**

Even though we used three layers, the network still behaves like a linear classifier.



# 7. Neural Network WITH ReLU Activation

Now we add ReLU between the linear layers:

```text
Input
  ↓
Linear(2 → 16)
  ↓
ReLU
  ↓
Linear(16 → 16)
  ↓
ReLU
  ↓
Linear(16 → 1)
  ↓
Output
```

ReLU is:

\[
ReLU(x)=\max(0,x)
\]

The activation changes the representation after each layer.

Therefore, the composition is no longer equivalent to one linear transformation:

\[
W_3\,ReLU(W_2\,ReLU(W_1X+b_1)+b_2)+b_3
\]

This can model **non-linear relationships**.


In [ ]:

class ReLUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),

            nn.Linear(16, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


relu_model = ReLUNet().to(device)

print(relu_model)


## 8. Train the ReLU Network

In [ ]:

relu_losses = train_model(
    relu_model,
    X_train,
    y_train,
    epochs=1000,
    lr=0.01
)

print("Final training loss:", relu_losses[-1])


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(relu_losses)

plt.title("Training Loss — Network WITH ReLU")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.grid(alpha=0.25)
plt.show()


In [ ]:

relu_train_acc = evaluate_model(relu_model, X_train, y_train)
relu_test_acc = evaluate_model(relu_model, X_test, y_test)

print(f"ReLU Train Accuracy: {relu_train_acc:.4f}")
print(f"ReLU Test Accuracy : {relu_test_acc:.4f}")


## 9. Plot the Non-Linear Decision Boundary

In [ ]:

plot_decision_boundary(
    relu_model,
    X_test,
    y_test,
    "Decision Boundary — WITH ReLU"
)

plt.show()



### Observation

The ReLU network can create a **curved/nonlinear decision region** that follows the structure of the two moons.

This is the practical effect of adding nonlinear activation functions.

The network is no longer restricted to:

\[
w_1x_1+w_2x_2+b=0
\]

Instead, the sequence of linear transformations and ReLU activations creates a much more flexible function.


## 10. Compare Both Decision Boundaries

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_decision_boundary(
    linear_model,
    X_test,
    y_test,
    "WITHOUT Activation
Linear Boundary",
    ax=axes[0]
)

plot_decision_boundary(
    relu_model,
    X_test,
    y_test,
    "WITH ReLU
Non-Linear Boundary",
    ax=axes[1]
)

plt.tight_layout()
plt.show()


## 11. Accuracy Comparison

In [ ]:

models = ["Linear Only", "ReLU Network"]
accuracies = [linear_test_acc, relu_test_acc]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies)

plt.ylim(0, 1.05)
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy Comparison")

for bar, value in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        fontweight="bold"
    )

plt.grid(axis="y", alpha=0.25)
plt.show()



# 12. The Mathematical Reason

Consider two layers **without activation**:

\[
h=W_1x+b_1
\]

\[
y=W_2h+b_2
\]

Substitute the first equation:

\[
y=W_2(W_1x+b_1)+b_2
\]

\[
y=W_2W_1x+W_2b_1+b_2
\]

Define:

\[
W'=W_2W_1
\]

and:

\[
b'=W_2b_1+b_2
\]

Therefore:

\[
\boxed{y=W'x+b'}
\]

So multiple linear layers collapse into **one linear transformation**.

---

## With an activation function

Now consider:

\[
h=ReLU(W_1x+b_1)
\]

\[
y=W_2h+b_2
\]

Therefore:

\[
\boxed{y=W_2ReLU(W_1x+b_1)+b_2}
\]

Because ReLU is nonlinear, we **cannot collapse this into one linear transformation**.

That is the fundamental reason activation functions are necessary in deep neural networks.



# 13. Final Takeaway

### Without activation

```text
Linear → Linear → Linear
          ↓
     Still Linear
          ↓
 Straight decision boundary
          ↓
 Cannot properly capture the moons
```

### With activation

```text
Linear → ReLU → Linear → ReLU → Linear
                 ↓
            Non-linear
                 ↓
      Flexible decision boundary
                 ↓
    Captures complex relationships
```

## Interview-Level Conclusion

> **Activation functions introduce non-linearity into neural networks. Without them, stacking multiple linear layers is mathematically equivalent to a single linear layer, so the network can only learn linear relationships. By inserting nonlinear functions such as ReLU between layers, the network can learn complex nonlinear patterns and construct nonlinear decision boundaries.**

### The most important idea to remember

\[
\boxed{\text{Linear layers alone} \Rightarrow \text{Linear model}}
\]

\[
\boxed{\text{Linear + Nonlinear activation} \Rightarrow \text{Nonlinear model}}
\]



## 14. Optional Experiment

Try changing:

```python
noise=0.20
```

to:

```python
noise=0.05
```

or:

```python
noise=0.35
```

and retrain both models.

You can also change the hidden-layer size:

```python
nn.Linear(2, 16)
```

to:

```python
nn.Linear(2, 32)
```

The goal is to observe how model capacity and dataset difficulty affect the learned decision boundary.
